In [1]:
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/rakesh/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [5]:
import cv2
from ultralytics import YOLO

# Load the model
model = YOLO("yolov8n.pt")

# Open the video source (0 for built-in webcam, or 'path/to/video.mp4')
cap = cv2.VideoCapture(0)

while cap.isOpened():
    success, frame = cap.read()
    
    if success:
        # Run YOLO detection on the current frame
        # stream=True utilizes a generator for better memory management with video
        results = model(frame, stream=True)
        
        for r in results:
            # Get the annotated frame
            annotated_frame = r.plot()
            
            # Display the stream
            cv2.imshow("YOLO Real-Time Detection", annotated_frame)
            
        # Break the loop if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
    else:
        break

# Clean up
cap.release()
cv2.destroyAllWindows()


0: 480x640 3 persons, 1 chair, 16.5ms
Speed: 0.8ms preprocess, 16.5ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.4ms
Speed: 0.8ms preprocess, 14.4ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 16.0ms
Speed: 0.5ms preprocess, 16.0ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 13.8ms
Speed: 0.5ms preprocess, 13.8ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.3ms
Speed: 0.6ms preprocess, 14.3ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 14.0ms
Speed: 0.6ms preprocess, 14.0ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 4 persons, 1 chair, 15.7ms
Speed: 0.8ms preprocess, 15.7ms inference, 0.3ms postprocess per image at shape (1, 3, 480, 640)

0: 480x640 3 persons, 1 chair, 13.9ms
Speed: 0.

In [4]:
from ultralytics import YOLO

# 1. Load the YOLO detection model (automatically downloads the weights if not present)
# Replace 'yolov8n.pt' or 'yolo11n.pt' with your specific model version (e.g., n, s, m, l, x)
model = YOLO("yolov8n.pt")

# 2. Run inference on an image
# You can use a local path ('path/to/image.jpg') or a web URL
source = "image3.jpg"
results = model(source)

# 3. Process and display the results
for result in results:
    # Print detected bounding boxes, confidence scores, and class IDs to the console
    print("Boxes:", result.boxes.xyxy)      # [xmin, ymin, xmax, ymax]
    print("Confidences:", result.boxes.conf)  # Confidence scores
    print("Class IDs:", result.boxes.cls)      # Class indexes (e.g., 0 for person)
    
    # Show the image with bounding boxes drawn on top
    result.show()
    
    # Save the annotated image to the default directory (runs/detect/predict/)
    result.save(filename="detected_result.jpg")


image 1/1 /home/rakesh/GitRepo/Object-Detection-Models/image3.jpg: 384x640 1 banana, 1 apple, 24.0ms
Speed: 1.9ms preprocess, 24.0ms inference, 0.4ms postprocess per image at shape (1, 3, 384, 640)
Boxes: tensor([[152.8540,  43.6977, 252.2990, 136.3970],
        [ 18.4016,  30.5332, 143.0974, 134.3175]])
Confidences: tensor([0.9080, 0.8760])
Class IDs: tensor([47., 46.])


/usr/bin/eog: symbol lookup error: /snap/core20/current/lib/x86_64-linux-gnu/libpthread.so.0: undefined symbol: __libc_pthread_init, version GLIBC_PRIVATE


#### **Custom Folder Inference Pipeline**

**Key Optimizations in this Pipeline**

- **stream=True**: Crucial for large datasets. Instead of loading all images and their predictions into your system RAM simultaneously, it streams them iteratively.
- **result.path**: Allows you to dynamically grab the file names from your input source so your outputs stay systematically organized.
- **result.plot()**: Generates a standard NumPy array with the drawn bounding boxes if you plan to do further image processing (like cropping or adding custom overlays with OpenCV).

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO

# 1. Initialize the model
# Using yolo26n as referenced in the Ultralytics suite
model = YOLO("yolo26n.pt") 

# 2. Define your folder paths
input_folder = "path/to/your/input_images"
output_folder = "path/to/your/saved_results"

# Ensure the output directory exists
os.makedirs(output_folder, exist_ok=True)

# 3. Run inference on the entire folder
# stream=True processes images one by one (generator mode) to prevent memory overload
results = model(input_folder, stream=True)

# 4. Iterate through the results and process each image
for result in results:
    # Get the original filename (e.g., 'photo1.jpg')
    original_path = Path(result.path)
    filename = original_path.name
    
    print(f"Processing: {filename}...")
    
    # --- Example A: Extracting Bounding Boxes to use in your code ---
    boxes = result.boxes.xyxy    # Coordinates: [xmin, ymin, xmax, ymax]
    scores = result.boxes.conf   # Confidence scores
    clss = result.boxes.cls      # Class IDs
    
    for box, score, cls in zip(boxes, scores, clss):
        class_name = model.names[int(cls)]
        print(f"  - Found {class_name} with confidence {score:.2f} at {box.tolist()}")

    # --- Example B: Saving the visual bounding boxes to your custom folder ---
    # Plot drawing onto the image frame
    annotated_frame = result.plot() 
    
    # Save the visual frame directly to your specific output folder
    save_path = os.path.join(output_folder, f"detected_{filename}")
    result.save(filename=save_path)

print(f"\n Pipeline complete! All annotated images saved to: {output_folder}")

In [6]:
import os
import csv
import cv2
from pathlib import Path
from ultralytics import YOLO

# 1. Initialize the model
model = YOLO("yolo26n.pt") 

# 2. Define your folder paths
input_folder = "Datasets/Custom/input_images"
output_base = "Datasets/Custom/saved_results"
crops_folder = os.path.join(output_base, "cropped_objects")
csv_path = os.path.join(output_base, "detection_results.csv")

# Ensure base directories exist
os.makedirs(crops_folder, exist_ok=True)
os.makedirs(output_base, exist_ok=True)

# 3. Open CSV file to log data dynamically
with open(csv_path, mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    # Write the column headers
    writer.writerow(["Filename", "Class Name", "Confidence", "Xmin", "Ymin", "Xmax", "Ymax"])

    print("Starting pipeline processing...")
    
    # 4. Stream batch inference on the entire folder
    results = model(input_folder, stream=True)

    for result in results:
        # Load the raw original image for precise cropping
        orig_img = result.orig_img
        filename = Path(result.path).name
        
        # Check if objects were detected
        if len(result.boxes) == 0:
            continue
            
        print(f"Processing detections in: {filename}")
        
        # Extract coordinates, scores, and class mappings
        # .xyxy coordinates are floats, .cpu().numpy() moves them off GPU if necessary
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        clss = result.boxes.cls.cpu().numpy()
        
        for i, (box, score, cls) in enumerate(zip(boxes, scores, clss)):
            class_name = model.names[int(cls)]
            xmin, ymin, xmax, ymax = map(int, box) # Convert to integers for indexing pixel coordinates
            
            # --- STEP A: LOG TO CSV ---
            writer.writerow([filename, class_name, f"{score:.4f}", xmin, ymin, xmax, ymax])
            
            # --- STEP B: CROP AND ORGANIZE CLASS SUBFOLDERS ---
            # Create a target folder specific to this object type (e.g., cropped_objects/person/)
            class_dir = os.path.join(crops_folder, class_name)
            os.makedirs(class_dir, exist_ok=True)
            
            # Crop the target frame using standard NumPy slicing [ymin:ymax, xmin:xmax]
            cropped_obj = orig_img[ymin:ymax, xmin:xmax]
            
            # Guard against invalid or empty bounding box boundaries
            if cropped_obj.size > 0:
                crop_filename = f"{Path(filename).stem}_{class_name}_{i}.jpg"
                crop_save_path = os.path.join(class_dir, crop_filename)
                
                # Save out the individual piece using OpenCV
                cv2.imwrite(crop_save_path, cropped_obj)

print(f"\nPipeline processing complete!")
print(f" Data log exported to: {csv_path}")
print(f" Cropped segments grouped under: {crops_folder}")


Starting pipeline processing...

image 1/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5244.HEIC: 640x480 2 persons, 1 couch, 27.2ms
Processing detections in: IMG_5244.HEIC
image 2/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5245.HEIC: 640x480 4 persons, 18.3ms
Processing detections in: IMG_5245.HEIC
image 3/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5246.HEIC: 640x480 1 person, 1 chair, 13.8ms
Processing detections in: IMG_5246.HEIC
image 4/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5247.HEIC: 640x480 1 person, 1 couch, 18.8ms
Processing detections in: IMG_5247.HEIC
image 5/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5248.HEIC: 640x480 1 person, 1 chair, 18.7ms
Processing detections in: IMG_5248.HEIC
image 6/36 /home/rakesh/GitRepo/Object-Detection-Models/Datasets/Custom/input_images/IMG_5249.HEIC: 640x

In [2]:
import os
import cv2
from ultralytics import YOLO

# 1. Paths Configuration
INPUT_DIR = "Datasets/face_database"           # Raw database folder
OUTPUT_DIR = "Datasets/face_database_cropped"  # Target destination

# Load YOLOv26 
model = YOLO("yolo26n.pt") 

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting extraction pipeline...")

# 2. Iterate through directories
for person_name in os.listdir(INPUT_DIR):
    person_input_path = os.path.join(INPUT_DIR, person_name)
    
    if not os.path.isdir(person_input_path):
        continue
        
    person_output_path = os.path.join(OUTPUT_DIR, person_name)
    os.makedirs(person_output_path, exist_ok=True)
    
    print(f"\n📂 Processing folder: {person_name}")
    
    # Loop through all images of this person
    for img_name in os.listdir(person_input_path):
        # Support standard extensions (assuming HEIC files are now converted to .jpg)
        if not img_name.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp')):
            continue
            
        img_path = os.path.join(person_input_path, img_name)
        image = cv2.imread(img_path)
        
        if image is None:
            print(f"  ❌ Could not read image matrix for: {img_name}")
            continue
            
        height, width, _ = image.shape
        
        # 3. YOLO Inference
        results = model(image, conf=0.25, verbose=False)
        
        best_box = None
        highest_conf = 0.0
        
        for result in results:
            boxes = result.boxes.cpu().numpy()
            for box in boxes:
                # FIX: Access the first element [0] before turning into a python float
                conf = float(box.conf[0])
                
                # We specifically look for the 'person' class (ID 0) to avoid cropping background items
                class_id = int(box.cls[0])
                
                if class_id == 0 and conf > highest_conf:
                    highest_conf = conf
                    best_box = box.xyxy[0] # Grab the specific coordinate array
        
        # 4. Save if found
        if best_box is not None:
            x1, y1, x2, y2 = map(int, best_box)
            
            # Bound coordinates to image dimensions
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(width, x2), min(height, y2)
            
            # Slice image array
            cropped_face = image[y1:y2, x1:x2]
            
            # Verify the crop isn't empty before writing to disk
            if cropped_face.size > 0:
                output_img_path = os.path.join(person_output_path, img_name)
                cv2.imwrite(output_img_path, cropped_face)
                print(f"  ✅ Cropped & Saved: {img_name}")
            else:
                print(f"  ⚠️ Invalid crop size for: {img_name}")
        else:
            print(f"  ❓ No person detected by model in: {img_name}")

print(f"\n Pipeline finished processing. Please check: '{OUTPUT_DIR}'")

Starting extraction pipeline...

📂 Processing folder: 07
  ✅ Cropped & Saved: IMG_5281.jpg
  ✅ Cropped & Saved: IMG_5279.jpg
  ✅ Cropped & Saved: IMG_5277.jpg
  ✅ Cropped & Saved: IMG_5278.jpg
  ✅ Cropped & Saved: IMG_5280.jpg

📂 Processing folder: 06
  ✅ Cropped & Saved: IMG_5274.jpg
  ✅ Cropped & Saved: IMG_5275.jpg
  ✅ Cropped & Saved: IMG_5270.jpg
  ✅ Cropped & Saved: IMG_5273.jpg
  ✅ Cropped & Saved: IMG_5271.jpg
  ✅ Cropped & Saved: IMG_5272.jpg

📂 Processing folder: 04
  ✅ Cropped & Saved: IMG_5260.jpg
  ✅ Cropped & Saved: IMG_5261.jpg
  ✅ Cropped & Saved: IMG_5259.jpg
  ✅ Cropped & Saved: IMG_5263.jpg

📂 Processing folder: 02
  ✅ Cropped & Saved: IMG_5250.jpg
  ✅ Cropped & Saved: IMG_5252.jpg
  ✅ Cropped & Saved: IMG_5251.jpg
  ✅ Cropped & Saved: IMG_5253.jpg

📂 Processing folder: 05
  ✅ Cropped & Saved: IMG_5269.jpg
  ✅ Cropped & Saved: IMG_5267.jpg
  ✅ Cropped & Saved: IMG_5265.jpg
  ✅ Cropped & Saved: IMG_5268.jpg
  ✅ Cropped & Saved: IMG_5266.jpg
  ✅ Cropped & Saved: IMG_52

In [ ]:
# find . -type f -iname "*.heic" -exec sh -c 'convert "$1" "${1%.*}.jpg" && rm "$1"' _ {} \;